In [ ]:
from pathlib import Path

import pandas as pd

from steam_api_analyzer import fetch_genres, get_genres, get_owned_games

In [ ]:
games = get_owned_games()

In [ ]:
print(games.head(5))

In [ ]:
games.info()

In [ ]:
# new column with hours played
games["hours_played"] = (games["playtime_forever"] / 60).round(1)
print(games[["name", "hours_played"]].head(5))

In [ ]:
# change of rtime_last_played
games["last_played"] = pd.to_datetime(games["rtime_last_played"], unit="s")
games["last_played"] = games["last_played"].replace(pd.Timestamp("1970-01-01 00:00:00"), pd.NaT)
games["days_since_last_played"] = (pd.Timestamp.now() - games["last_played"]).dt.days.astype(
    "Int64"
)

In [ ]:
# top 10:
top10 = games.sort_values("hours_played", ascending=False).head(10)
print(top10[["name", "hours_played"]])

In [ ]:
# Count and share of never launched games
never_launched = (games[games["playtime_forever"] == 0]).shape[0]
never_launched_share = never_launched / len(games)

print(f"{never_launched} games have never been launched,")
print(f"which is {never_launched_share:.1%} of all games.")

In [ ]:
# Games not launched for the longest time and days since last played
least_recently_played = games[games["playtime_forever"] != 0].sort_values(
    "last_played", ascending=True
)
least_recently_played[["name", "last_played", "hours_played", "days_since_last_played"]].head(10)

In [ ]:
# Genres of the most played game
top_appid = top10.iloc[0]["appid"]
print(f"Top played game: {top10.iloc[0]['name']}")

In [ ]:
top_genres = fetch_genres(top_appid)
print(top_genres)

In [ ]:
# App not in the store returns an empty list.
fetch_genres(1)

In [ ]:
# Genres for the whole library (cached in data/)
genres = get_genres(games["appid"], Path("../data/genres_cache.json"))
genres

In [ ]:
# Merge genres with the library data
games["genres"] = games["appid"].map(genres)
games[["name", "hours_played", "genres"]].head(10)

In [ ]:
# Games missing from the genre cache (NaN after .map, should be 0)
genres_is_nan = games["genres"].isna().sum()
print(f"{genres_is_nan} games are missing from the genre cache.")

In [ ]:
# Games without genres in the store (empty list: removed from the store or no genres field)
no_genres = games[games["genres"].str.len() == 0]
no_genres_share = len(no_genres) / len(games)

print(f"{len(no_genres)} games have no genres assigned,")
print(f"which is {no_genres_share:.1%} of all games.")
no_genres[["appid", "name"]]

In [ ]:
# Explode genres to have one genre per row
games_exploded = games.explode("genres")
games_exploded[["name", "hours_played", "genres"]]

In [ ]:
# Group by genre and sum hours played
games_exploded.groupby("genres")["hours_played"].sum().sort_values(ascending=False)

In [ ]:
# Group by genre and count games
games_exploded.groupby("genres").size().sort_values(ascending=False)

In [ ]:
# test
games_exploded[games_exploded["genres"] == "Video Production"]

In [ ]:
# test
games_exploded["genres"].isna().sum()